# Image Segmentation Classification

Every instance is a 3x3 region from one of seven outdoor images, described by 19 measurements of colour, edges and texture. Label each region as brickface, sky, foliage, cement, window, path or grass — the pixel-level labelling step behind image understanding.

In [1]:
import io, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def download(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=180) as r:
        return r.read()
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# UCI Statlog (Image Segmentation) (id 147) — the UCI image-segmentation
# dataset in its Statlog form, 2310 regions, 19 features, 7 balanced classes.
df = pd.read_csv(io.BytesIO(download('https://archive.ics.uci.edu/static/public/147/data.csv')))
print(df.shape)
print('Regions per class code (1=brickface, 2=sky, 3=foliage, 4=cement, 5=window, 6=path, 7=grass):')
print(df['class'].value_counts().sort_index())
df.head(3)


(2310, 20)
Regions per class code (1=brickface, 2=sky, 3=foliage, 4=cement, 5=window, 6=path, 7=grass):
class
1    330
2    330
3    330
4    330
5    330
6    330
7    330
Name: count, dtype: int64


,region-centroid-col,region-centroid-row,region-pixel-count,short-line-density-5,short-line-density-2,vedge-mean,vedge-sd,hedge-mean,hedge-sd,intensity-mean,rawred-mean,rawblue-mean,rawgreen-mean,exred-mean,exblue-mean,exgreen-mean,value-mean,saturation-mean,hue-mean,class
0,218,178,9,0.111111,0.0,0.833333,0.547722,1.111109,0.544331,59.629630,52.444443,75.222220,51.222220,-21.555555,46.77778,-25.222221,75.222220,0.318996,-2.040554,6
1,113,130,9,0.000000,0.0,0.277778,0.250924,0.333333,0.365148,0.888889,0.000000,2.555556,0.111111,-2.666667,5.00000,-2.333333,2.555556,1.000000,-2.123254,3
2,202,41,9,0.000000,0.0,0.944448,0.772202,1.111112,1.025597,123.037040,111.888885,139.777790,117.444440,-33.444443,50.22222,-16.777779,139.777790,0.199347,-2.299918,2


In [2]:

# region-pixel-count is constant (9, every region is 3x3) and carries no
# signal, so it is dropped — the same constant-feature rule used for
# Ionosphere in an earlier batch.
feats = [c for c in df.columns if c not in ('class', 'region-pixel-count')]
names = {1: 'brickface', 2: 'sky', 3: 'foliage', 4: 'cement', 5: 'window', 6: 'path', 7: 'grass'}
X = df[feats]; y = df['class'].map(names)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(Xtr, ytr)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
acc_rf = accuracy_score(yte, rf.predict(Xte)); acc_lr = accuracy_score(yte, lr.predict(Xte))
print(f'Random Forest accuracy: {acc_rf:.4f}')
print(f'Logistic Regression accuracy: {acc_lr:.4f}')
print(classification_report(yte, rf.predict(Xte)))


Random Forest accuracy: 0.9689
Logistic Regression accuracy: 0.9343
              precision    recall  f1-score   support

   brickface       1.00      0.99      0.99        82
      cement       0.97      0.93      0.95        82
     foliage       0.94      0.94      0.94        83
       grass       1.00      0.98      0.99        83
        path       1.00      1.00      1.00        83
         sky       1.00      1.00      1.00        82
      window       0.88      0.95      0.91        83

    accuracy                           0.97       578
   macro avg       0.97      0.97      0.97       578
weighted avg       0.97      0.97      0.97       578



In [3]:
imp = pd.Series(rf.feature_importances_, index=feats).sort_values(ascending=False)
print(imp)
imp.plot(kind='barh', title='Which region measurements separate the scenes?')
plt.tight_layout()
plt.show()
winner = rf if acc_rf >= acc_lr else lr
winner_name = 'Random Forest' if acc_rf >= acc_lr else 'Logistic Regression'
print(f'Winner: {winner_name}')
cm = confusion_matrix(yte, winner.predict(Xte), labels=sorted(y.unique()))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=sorted(y.unique()))
disp.plot(xticks_rotation=45)
plt.title(f'Confusion matrix - {winner_name}')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()


hue-mean                0.172130
region-centroid-row     0.145003
exgreen-mean            0.097980
rawred-mean             0.093415
intensity-mean          0.082712
rawblue-mean            0.066089
value-mean              0.063817
rawgreen-mean           0.056419
saturation-mean         0.049866
exred-mean              0.042444
exblue-mean             0.042206
hedge-mean              0.026970
region-centroid-col     0.024343
hedge-sd                0.016258
vedge-mean              0.010869
vedge-sd                0.008436
short-line-density-5    0.000546
short-line-density-2    0.000497
dtype: float64


Winner: Random Forest
